# Workshop Databricks - Telefonica Onboarding

## Bem-vindo à Plataforma Databricks! 🚀

Este workshop foi desenvolvido para apresentar as principais funcionalidades da plataforma Databricks, com foco em:

* **Unity Catalog**: Governança de dados unificada e gerenciamento de metadados
* **Delta Lake**: Formato de armazenamento com recursos avançados como Time Travel
* **Dashboards & Genie Spaces**: Visualização e análise de dados com IA

---

### Agenda do Workshop

1. **Configuração Inicial** - Criação de catálogo e schema no Unity Catalog
2. **Dados de Demonstração** - Geração de dataset simulado de telecomunicações
3. **Unity Catalog em Ação** - Exploração de recursos de governança
4. **Delta Lake Time Travel** - Versionamento e consultas históricas
5. **Dashboard & Genie** - Criação de visualizações e Genie Space

## 1. Configuração Inicial - Unity Catalog

O **Unity Catalog** é a solução de governança de dados da Databricks que fornece:

* Gerenciamento centralizado de metadados
* Controle de acesso granular (tabelas, colunas, linhas)
* Auditoria e linhagem de dados
* Compartilhamento seguro de dados entre organizações

### Estrutura Hierárquica

```
Catalog (Catálogo)
  └── Schema (Esquema/Database)
      └── Tables, Views, Functions
```

Vamos criar nossa estrutura para o workshop:

In [0]:
# Configuração do ambiente
import random
import string
from datetime import datetime, timedelta
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Gerar um identificador único para evitar conflitos
user_id = spark.sql("SELECT current_user()").collect()[0][0].split('@')[0].replace('.', '_')
timestamp_id = datetime.now().strftime("%Y%m%d")

# Nomes do catálogo e schema
catalog_name = f"workshop_telefonica_{user_id}"
schema_name = "telecom_data"

print(f"📦 Catálogo: {catalog_name}")
print(f"📂 Schema: {schema_name}")
print(f"\n🔧 Criando estrutura no Unity Catalog...")

# Criar catálogo (se não existir)
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"USE CATALOG {catalog_name}")

# Criar schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name} COMMENT 'Dados de telecomunicações para workshop'")
spark.sql(f"USE SCHEMA {schema_name}")

print(f"\n✅ Estrutura criada com sucesso!")
print(f"\n💡 Você pode explorar no Data Explorer: {catalog_name}.{schema_name}")

📦 Catálogo: workshop_telefonica_reinaldorcosta
📂 Schema: telecom_data

🔧 Criando estrutura no Unity Catalog...

✅ Estrutura criada com sucesso!

💡 Você pode explorar no Data Explorer: workshop_telefonica_reinaldorcosta.telecom_data


## 2. Geração de Dados de Demonstração

Vamos criar um dataset simulado de telecomunicações com:

* **Clientes**: Informações de assinantes
* **Planos**: Tipos de planos oferecidos
* **Uso de Dados**: Registros de consumo diário
* **Receita**: Transações e faturamento

Estes dados serão armazenados como **Delta Tables** no Unity Catalog.

In [0]:
# Gerar tabela de clientes
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DoubleType

print("👥 Gerando dados de clientes...")

# Dados de clientes
clientes_data = []
for i in range(1, 1001):
    cliente_id = f"CLI{i:05d}"
    nome = f"Cliente {i}"
    segmento = random.choice(["Residencial", "Empresarial", "Premium"])
    cidade = random.choice(["São Paulo", "Rio de Janeiro", "Brasília", "Salvador", "Belo Horizonte"])
    data_cadastro = (datetime.now() - timedelta(days=random.randint(30, 1095))).date()
    
    clientes_data.append((cliente_id, nome, segmento, cidade, data_cadastro))

schema_clientes = StructType([
    StructField("cliente_id", StringType(), False),
    StructField("nome", StringType(), False),
    StructField("segmento", StringType(), False),
    StructField("cidade", StringType(), False),
    StructField("data_cadastro", DateType(), False)
])

df_clientes = spark.createDataFrame(clientes_data, schema_clientes)

# Salvar como Delta Table
df_clientes.write.format("delta").mode("overwrite").saveAsTable("clientes")

print(f"✅ Tabela 'clientes' criada com {df_clientes.count()} registros")
display(df_clientes.limit(5))

👥 Gerando dados de clientes...
✅ Tabela 'clientes' criada com 1000 registros


cliente_id,nome,segmento,cidade,data_cadastro
CLI00001,Cliente 1,Empresarial,Rio de Janeiro,2023-07-10
CLI00002,Cliente 2,Empresarial,Belo Horizonte,2025-05-28
CLI00003,Cliente 3,Residencial,Belo Horizonte,2026-01-22
CLI00004,Cliente 4,Residencial,Rio de Janeiro,2024-10-26
CLI00005,Cliente 5,Residencial,Belo Horizonte,2025-06-15


In [0]:
# Gerar tabela de planos
print("📊 Gerando dados de planos...")

planos_data = [
    ("PLAN001", "Básico 5GB", "Pré-pago", 5, 29.90),
    ("PLAN002", "Intermediário 15GB", "Pós-pago", 15, 59.90),
    ("PLAN003", "Avançado 30GB", "Pós-pago", 30, 89.90),
    ("PLAN004", "Premium 50GB", "Pós-pago", 50, 129.90),
    ("PLAN005", "Ilimitado", "Pós-pago", 999, 199.90),
    ("PLAN006", "Empresarial 100GB", "Corporativo", 100, 299.90),
    ("PLAN007", "Empresarial Ilimitado", "Corporativo", 999, 499.90)
]

schema_planos = StructType([
    StructField("plano_id", StringType(), False),
    StructField("nome_plano", StringType(), False),
    StructField("tipo", StringType(), False),
    StructField("franquia_gb", IntegerType(), False),
    StructField("valor_mensal", DoubleType(), False)
])

df_planos = spark.createDataFrame(planos_data, schema_planos)
df_planos.write.format("delta").mode("overwrite").saveAsTable("planos")

print(f"✅ Tabela 'planos' criada com {df_planos.count()} registros")
display(df_planos)

📊 Gerando dados de planos...
✅ Tabela 'planos' criada com 7 registros


plano_id,nome_plano,tipo,franquia_gb,valor_mensal
PLAN001,Básico 5GB,Pré-pago,5,29.9
PLAN002,Intermediário 15GB,Pós-pago,15,59.9
PLAN003,Avançado 30GB,Pós-pago,30,89.9
PLAN004,Premium 50GB,Pós-pago,50,129.9
PLAN005,Ilimitado,Pós-pago,999,199.9
PLAN006,Empresarial 100GB,Corporativo,100,299.9
PLAN007,Empresarial Ilimitado,Corporativo,999,499.9


In [0]:
# Gerar dados de uso diário (90 dias)
print("📱 Gerando dados de uso de dados...")

uso_data = []
data_inicio = datetime.now() - timedelta(days=90)

# Selecionar amostra de clientes para gerar uso
clientes_sample = [f"CLI{i:05d}" for i in range(1, 501)]  # 500 clientes
planos_ids = ["PLAN001", "PLAN002", "PLAN003", "PLAN004", "PLAN005", "PLAN006", "PLAN007"]

# Use Python's built-in round (not PySpark's round)
import builtins

for cliente_id in clientes_sample:
    plano_id = random.choice(planos_ids)
    
    # Gerar 90 dias de uso
    for dia in range(90):
        data_uso = (data_inicio + timedelta(days=dia)).date()
        
        # Consumo varia por tipo de plano
        if "Ilimitado" in plano_id or plano_id in ["PLAN005", "PLAN007"]:
            consumo_gb = builtins.round(random.uniform(5, 25), 2)
        else:
            consumo_gb = builtins.round(random.uniform(0.5, 8), 2)
        
        minutos_voz = random.randint(10, 300)
        sms_enviados = random.randint(0, 50)
        
        uso_data.append((cliente_id, plano_id, data_uso, consumo_gb, minutos_voz, sms_enviados))

schema_uso = StructType([
    StructField("cliente_id", StringType(), False),
    StructField("plano_id", StringType(), False),
    StructField("data_uso", DateType(), False),
    StructField("consumo_gb", DoubleType(), False),
    StructField("minutos_voz", IntegerType(), False),
    StructField("sms_enviados", IntegerType(), False)
])

df_uso = spark.createDataFrame(uso_data, schema_uso)
df_uso.write.format("delta").mode("overwrite").saveAsTable("uso_diario")

print(f"✅ Tabela 'uso_diario' criada com {df_uso.count():,} registros")
print(f"\n📅 Período: {data_inicio.date()} até {datetime.now().date()}")
display(df_uso.limit(10))

📱 Gerando dados de uso de dados...
✅ Tabela 'uso_diario' criada com 45,000 registros

📅 Período: 2026-02-12 até 2026-05-13


cliente_id,plano_id,data_uso,consumo_gb,minutos_voz,sms_enviados
CLI00001,PLAN006,2026-02-12,2.2,244,49
CLI00001,PLAN006,2026-02-13,7.28,244,28
CLI00001,PLAN006,2026-02-14,2.97,48,8
CLI00001,PLAN006,2026-02-15,3.98,208,11
CLI00001,PLAN006,2026-02-16,6.41,291,29
CLI00001,PLAN006,2026-02-17,1.76,145,34
CLI00001,PLAN006,2026-02-18,7.66,263,40
CLI00001,PLAN006,2026-02-19,1.39,261,36
CLI00001,PLAN006,2026-02-20,3.02,182,17
CLI00001,PLAN006,2026-02-21,5.84,25,30


In [0]:
# Gerar dados de receita mensal
print("💰 Gerando dados de receita...")

# Use Python's built-in round (not PySpark's round)
import builtins

receita_data = []

# Gerar receita para os últimos 3 meses
for mes_offset in range(3):
    data_ref = datetime.now() - timedelta(days=30 * mes_offset)
    mes_ano = data_ref.strftime("%Y-%m")
    
    for cliente_id in clientes_sample:
        plano_id = random.choice(planos_ids)
        
        # Buscar valor do plano
        valor_base = next((p[4] for p in planos_data if p[0] == plano_id), 50.0)
        
        # Adicionar variações (excedentes, descontos, etc)
        valor_excedente = builtins.round(random.uniform(0, 30), 2) if random.random() > 0.7 else 0
        desconto = builtins.round(random.uniform(0, 20), 2) if random.random() > 0.8 else 0
        
        valor_total = builtins.round(valor_base + valor_excedente - desconto, 2)
        status_pagamento = random.choice(["Pago", "Pago", "Pago", "Pendente", "Atrasado"])
        
        data_vencimento = data_ref.replace(day=10).date()
        data_pagamento = (data_ref + timedelta(days=random.randint(-5, 15))).date() if status_pagamento == "Pago" else None
        
        receita_data.append((cliente_id, plano_id, mes_ano, data_vencimento, valor_base, 
                           valor_excedente, desconto, valor_total, status_pagamento, data_pagamento))

schema_receita = StructType([
    StructField("cliente_id", StringType(), False),
    StructField("plano_id", StringType(), False),
    StructField("mes_referencia", StringType(), False),
    StructField("data_vencimento", DateType(), False),
    StructField("valor_base", DoubleType(), False),
    StructField("valor_excedente", DoubleType(), False),
    StructField("desconto", DoubleType(), False),
    StructField("valor_total", DoubleType(), False),
    StructField("status_pagamento", StringType(), False),
    StructField("data_pagamento", DateType(), True)
])

df_receita = spark.createDataFrame(receita_data, schema_receita)
df_receita.write.format("delta").mode("overwrite").saveAsTable("receita_mensal")

print(f"✅ Tabela 'receita_mensal' criada com {df_receita.count():,} registros")
display(df_receita.limit(10))

💰 Gerando dados de receita...
✅ Tabela 'receita_mensal' criada com 1,500 registros


cliente_id,plano_id,mes_referencia,data_vencimento,valor_base,valor_excedente,desconto,valor_total,status_pagamento,data_pagamento
CLI00001,PLAN003,2026-05,2026-05-10,89.9,0.0,11.33,78.57,Pago,2026-05-11
CLI00002,PLAN007,2026-05,2026-05-10,499.9,0.0,0.0,499.9,Pago,2026-05-28
CLI00003,PLAN004,2026-05,2026-05-10,129.9,0.0,0.0,129.9,Atrasado,null
CLI00004,PLAN006,2026-05,2026-05-10,299.9,0.0,0.0,299.9,Pago,2026-05-12
CLI00005,PLAN003,2026-05,2026-05-10,89.9,0.0,0.0,89.9,Atrasado,null
CLI00006,PLAN004,2026-05,2026-05-10,129.9,0.0,0.0,129.9,Pendente,null
CLI00007,PLAN005,2026-05,2026-05-10,199.9,0.0,0.19,199.71,Pago,2026-05-28
CLI00008,PLAN002,2026-05,2026-05-10,59.9,0.0,0.0,59.9,Pago,2026-05-27
CLI00009,PLAN005,2026-05,2026-05-10,199.9,0.0,0.0,199.9,Pago,2026-05-13
CLI00010,PLAN006,2026-05,2026-05-10,299.9,0.0,1.77,298.13,Pago,2026-05-23


In [0]:
# Resumo das tabelas criadas
print("\n" + "="*60)
print("🎉 DADOS CRIADOS COM SUCESSO!")
print("="*60)

tabelas = spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}").collect()

print(f"\n📁 Catálogo: {catalog_name}")
print(f"📂 Schema: {schema_name}")
print(f"\n📊 Tabelas criadas:\n")

for tabela in tabelas:
    nome_tabela = tabela.tableName
    count = spark.table(f"{catalog_name}.{schema_name}.{nome_tabela}").count()
    print(f"  • {nome_tabela}: {count:,} registros")

print(f"\n🔗 Acesse no Data Explorer: {catalog_name}.{schema_name}")
print("\n➡️ Próximos passos: Explorar recursos do Unity Catalog e Delta Lake!")


🎉 DADOS CRIADOS COM SUCESSO!

📁 Catálogo: workshop_telefonica_reinaldorcosta
📂 Schema: telecom_data

📊 Tabelas criadas:

  • clientes: 1,000 registros
  • planos: 7 registros
  • receita_mensal: 1,500 registros
  • uso_diario: 45,000 registros

🔗 Acesse no Data Explorer: workshop_telefonica_reinaldorcosta.telecom_data

➡️ Próximos passos: Explorar recursos do Unity Catalog e Delta Lake!


## 3. Unity Catalog em Ação

Vamos explorar os recursos de governança e gerenciamento de metadados do Unity Catalog:

### Recursos que vamos demonstrar:

* **Metadados Enriquecidos**: Comentários e documentação de tabelas e colunas
* **Propriedades de Tabelas**: Informações sobre formato, localização e estatísticas
* **Linhagem de Dados**: Rastreamento de origem e uso dos dados
* **Controle de Acesso**: Permissões granulares (demonstração conceitual)
* **Auditoria**: Histórico de operações

In [0]:
%sql
-- Adicionar comentários às tabelas para documentação
COMMENT ON TABLE clientes IS 'Cadastro de clientes da Telefonica - contém informações de assinantes e segmentação';
COMMENT ON TABLE planos IS 'Catálogo de planos oferecidos - inclui franquias e preços';
COMMENT ON TABLE uso_diario IS 'Registros diários de consumo de dados, voz e SMS por cliente';
COMMENT ON TABLE receita_mensal IS 'Faturamento mensal por cliente - inclui valores base, excedentes e status de pagamento';

-- Adicionar comentários em colunas específicas
ALTER TABLE clientes ALTER COLUMN cliente_id COMMENT 'Identificador único do cliente (formato: CLI00000)';
ALTER TABLE clientes ALTER COLUMN segmento COMMENT 'Segmentação: Residencial, Empresarial ou Premium';

ALTER TABLE uso_diario ALTER COLUMN consumo_gb COMMENT 'Consumo de dados em Gigabytes';
ALTER TABLE uso_diario ALTER COLUMN minutos_voz COMMENT 'Minutos de ligações de voz';

SELECT '✅ Comentários adicionados com sucesso!' as status

status
✅ Comentários adicionados com sucesso!


In [0]:
%sql
-- Visualizar metadados detalhados de uma tabela
DESCRIBE EXTENDED clientes

col_name,data_type,comment
cliente_id,string,Identificador único do cliente (formato: CLI00000)
nome,string,null
segmento,string,"Segmentação: Residencial, Empresarial ou Premium"
cidade,string,null
data_cadastro,date,null
,,
# Delta Statistics Columns,,
Column Names,"segmento, cidade, cliente_id, nome, data_cadastro",
Column Selection Method,first-32,
,,


In [0]:
# Explorar propriedades das tabelas Delta
print("🔍 Explorando propriedades das tabelas Delta\n")

tabelas = ["clientes", "planos", "uso_diario", "receita_mensal"]

for tabela in tabelas:
    print(f"\n{'='*60}")
    print(f"📊 Tabela: {tabela}")
    print(f"{'='*60}")
    
    # Obter detalhes da tabela
    detalhes = spark.sql(f"DESCRIBE DETAIL {catalog_name}.{schema_name}.{tabela}").collect()[0]
    
    print(f"  • Formato: {detalhes.format}")
    print(f"  • Localização: {detalhes.location}")
    print(f"  • Número de arquivos: {detalhes.numFiles}")
    print(f"  • Tamanho: {detalhes.sizeInBytes / 1024 / 1024:.2f} MB")
    print(f"  • Última modificação: {detalhes.lastModified}")
    
print(f"\n\n💡 Todas as tabelas estão no formato Delta Lake!")

🔍 Explorando propriedades das tabelas Delta


📊 Tabela: clientes
  • Formato: delta
  • Localização: 
  • Número de arquivos: 1
  • Tamanho: 0.01 MB
  • Última modificação: 2026-05-13 15:21:02

📊 Tabela: planos
  • Formato: delta
  • Localização: 
  • Número de arquivos: 1
  • Tamanho: 0.00 MB
  • Última modificação: 2026-05-13 15:20:57

📊 Tabela: uso_diario
  • Formato: delta
  • Localização: 
  • Número de arquivos: 8
  • Tamanho: 0.20 MB
  • Última modificação: 2026-05-13 15:21:04

📊 Tabela: receita_mensal
  • Formato: delta
  • Localização: 
  • Número de arquivos: 2
  • Tamanho: 0.02 MB
  • Última modificação: 2026-05-13 15:21:00


💡 Todas as tabelas estão no formato Delta Lake!


In [0]:
%sql
-- Consultar o Information Schema do Unity Catalog
-- Listar todas as tabelas com seus comentários

SELECT 
  table_name,
  table_type,
  comment,
  created
FROM system.information_schema.tables
WHERE table_catalog = current_catalog()
  AND table_schema = current_schema()
ORDER BY table_name

table_name,table_type,comment,created
clientes,MANAGED,Cadastro de clientes da Telefonica - contém informações de assinantes e segmentação,2026-05-13T15:20:25.577Z
planos,MANAGED,Catálogo de planos oferecidos - inclui franquias e preços,2026-05-13T15:20:38.403Z
receita_mensal,MANAGED,"Faturamento mensal por cliente - inclui valores base, excedentes e status de pagamento",2026-05-13T15:20:50.091Z
uso_diario,MANAGED,"Registros diários de consumo de dados, voz e SMS por cliente",2026-05-13T15:20:45.190Z


In [0]:
%sql
-- Visualizar metadados de colunas com comentários

SELECT 
  table_name,
  column_name,
  data_type,
  comment
FROM system.information_schema.columns
WHERE table_catalog = current_catalog()
  AND table_schema = current_schema()
  AND table_name = 'clientes'
ORDER BY ordinal_position

table_name,column_name,data_type,comment
clientes,cliente_id,STRING,Identificador único do cliente (formato: CLI00000)
clientes,nome,STRING,null
clientes,segmento,STRING,"Segmentação: Residencial, Empresarial ou Premium"
clientes,cidade,STRING,null
clientes,data_cadastro,DATE,null


### 🎯 Benefícios do Unity Catalog Demonstrados

✅ **Metadados Centralizados**: Todos os comentários e documentações em um só lugar

✅ **Descoberta de Dados**: Fácil localização de tabelas e colunas através do Data Explorer

✅ **Governança**: Controle granular de acesso (nível de catálogo, schema, tabela, coluna)

✅ **Auditoria**: Rastreamento completo de quem acessa e modifica os dados

✅ **Linhagem**: Visualização de origem e destino dos dados (disponível no Data Explorer)

---

💡 **Dica**: No Data Explorer, você pode visualizar graficamente a linhagem de dados, permissões e histórico de acesso!

## 4. Delta Lake Time Travel

O **Delta Lake** é um formato de armazenamento open-source que traz confiabilidade para data lakes:

### Principais Recursos:

* **ACID Transactions**: Garantia de consistência dos dados
* **Time Travel**: Acesso a versões históricas dos dados
* **Schema Evolution**: Evolução do schema sem quebrar pipelines
* **Audit History**: Histórico completo de todas as operações
* **Upserts e Deletes**: Operações DML eficientes

Vamos demonstrar o **Time Travel** fazendo modificações na tabela de clientes!

In [0]:
%sql
-- Visualizar histórico de versões da tabela
DESCRIBE HISTORY clientes

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-05-13T15:21:02.000Z,4010784151952347,reinaldorcosta@gmail.com,CHANGE COLUMN,"Map(column -> {""name"":""segmento"",""type"":""string"",""nullable"":true,""metadata"":{""comment"":""Segmentação: Residencial, Empresarial ou Premium""}})",null,List(4153158223177044),29c49b22-7ad1-48da-b3e3-5b3481b8e7a6,0513-151938-bd2extjs-v2n,2,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
2,2026-05-13T15:21:01.000Z,4010784151952347,reinaldorcosta@gmail.com,CHANGE COLUMN,"Map(column -> {""name"":""cliente_id"",""type"":""string"",""nullable"":true,""metadata"":{""comment"":""Identificador único do cliente (formato: CLI00000)""}})",null,List(4153158223177044),e4dfb2f1-0bd2-4262-bf9a-3465c9d59721,0513-151938-bd2extjs-v2n,1,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-05-13T15:20:56.000Z,4010784151952347,reinaldorcosta@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""comment"":""Cadastro de clientes da Telefonica - contém informações de assinantes e segmentação""})",null,List(4153158223177044),9b8bc872-2e04-449a-8bb8-ec45f1a80b8e,0513-151938-bd2extjs-v2n,0,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-05-13T15:20:26.000Z,4010784151952347,reinaldorcosta@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4153158223177044),e1126fad-adc6-4e82-a4b7-f38b15af8968,0513-151938-bd2extjs-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1000, numOutputBytes -> 6149)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


In [0]:
# Simular adição de novos clientes
print("🔄 Atualização 1: Adicionando novos clientes...\n")

novos_clientes = [
    ("CLI01001", "Maria Silva", "Premium", "São Paulo", datetime.now().date()),
    ("CLI01002", "João Santos", "Empresarial", "Rio de Janeiro", datetime.now().date()),
    ("CLI01003", "Ana Costa", "Residencial", "Brasília", datetime.now().date())
]

df_novos = spark.createDataFrame(novos_clientes, schema_clientes)
df_novos.write.format("delta").mode("append").saveAsTable("clientes")

count_atual = spark.table("clientes").count()
print(f"✅ 3 novos clientes adicionados!")
print(f"📊 Total de clientes agora: {count_atual}")

# Mostrar os novos clientes
print("\n🆕 Novos clientes:")
display(spark.sql("SELECT * FROM clientes WHERE cliente_id >= 'CLI01001' ORDER BY cliente_id"))

🔄 Atualização 1: Adicionando novos clientes...

✅ 3 novos clientes adicionados!
📊 Total de clientes agora: 1003

🆕 Novos clientes:


cliente_id,nome,segmento,cidade,data_cadastro
CLI01001,Maria Silva,Premium,São Paulo,2026-05-13
CLI01002,João Santos,Empresarial,Rio de Janeiro,2026-05-13
CLI01003,Ana Costa,Residencial,Brasília,2026-05-13


In [0]:
%sql
-- Atualizar segmento de alguns clientes
UPDATE clientes 
SET segmento = 'Premium'
WHERE cliente_id IN ('CLI00001', 'CLI00002', 'CLI00003')
  AND segmento != 'Premium';

SELECT '3 clientes atualizados para segmento Premium' as status

status
3 clientes atualizados para segmento Premium


In [0]:
%sql
-- Deletar um cliente de teste
DELETE FROM clientes WHERE cliente_id = 'CLI01003';

SELECT 'Cliente CLI01003 removido' as status

status
Cliente CLI01003 removido


In [0]:
%sql
-- Visualizar histórico atualizado com todas as operações
SELECT 
  version,
  timestamp,
  operation,
  operationParameters,
  operationMetrics
FROM (DESCRIBE HISTORY clientes)
ORDER BY version DESC
LIMIT 10

version,timestamp,operation,operationParameters,operationMetrics
6,2026-05-13T15:21:29.000Z,DELETE,"Map(predicate -> [""(cliente_id#16945 = CLI01003)""])","Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1555, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 979, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 575)"
5,2026-05-13T15:21:26.000Z,UPDATE,"Map(predicate -> [""(cliente_id#16572 IN (CLI00001,CLI00002,CLI00003) AND NOT (segmento#16574 = Premium))""])","Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3926, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1818, numAddedFiles -> 1, numUpdatedRows -> 3, numAddedBytes -> 1854, rewriteTimeMs -> 2066)"
4,2026-05-13T15:21:19.000Z,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])","Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1719)"
3,2026-05-13T15:21:02.000Z,CHANGE COLUMN,"Map(column -> {""name"":""segmento"",""type"":""string"",""nullable"":true,""metadata"":{""comment"":""Segmentação: Residencial, Empresarial ou Premium""}})",Map()
2,2026-05-13T15:21:01.000Z,CHANGE COLUMN,"Map(column -> {""name"":""cliente_id"",""type"":""string"",""nullable"":true,""metadata"":{""comment"":""Identificador único do cliente (formato: CLI00000)""}})",Map()
1,2026-05-13T15:20:56.000Z,SET TBLPROPERTIES,"Map(properties -> {""comment"":""Cadastro de clientes da Telefonica - contém informações de assinantes e segmentação""})",Map()
0,2026-05-13T15:20:26.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)","Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1000, numOutputBytes -> 6149)"


In [0]:
%sql
-- Consultar a versão original da tabela (versão 0)
SELECT 
  'Versão Original (v0)' as versao,
  COUNT(*) as total_clientes,
  COUNT(DISTINCT segmento) as segmentos_distintos
FROM clientes VERSION AS OF 0

versao,total_clientes,segmentos_distintos
Versão Original (v0),1000,3


In [0]:
# Comparar diferentes versões da tabela
print("🔍 Comparando versões da tabela de clientes\n")
print("="*70)

# Obter número de versões
history = spark.sql("DESCRIBE HISTORY clientes")
max_version = history.agg({"version": "max"}).collect()[0][0]

print(f"\n📊 Total de versões: {max_version + 1} (0 a {max_version})\n")

# Comparar versão inicial vs atual
v0_count = spark.sql("SELECT COUNT(*) as count FROM clientes VERSION AS OF 0").collect()[0][0]
v_current_count = spark.table("clientes").count()

print(f"Versão 0 (Original):  {v0_count} clientes")
print(f"Versão {max_version} (Atual):     {v_current_count} clientes")
print(f"Diferença:            {v_current_count - v0_count:+d} clientes")

print("\n" + "="*70)

# Mostrar clientes que foram promovidos para Premium
print("\n🌟 Clientes promovidos para Premium (comparando versões):\n")

query = """
SELECT 
  atual.cliente_id,
  atual.nome,
  original.segmento as segmento_original,
  atual.segmento as segmento_atual
FROM clientes atual
INNER JOIN clientes VERSION AS OF 0 original
  ON atual.cliente_id = original.cliente_id
WHERE atual.segmento = 'Premium' 
  AND original.segmento != 'Premium'
LIMIT 5
"""

display(spark.sql(query))

🔍 Comparando versões da tabela de clientes


📊 Total de versões: 8 (0 a 7)

Versão 0 (Original):  1000 clientes
Versão 7 (Atual):     1002 clientes
Diferença:            +2 clientes


🌟 Clientes promovidos para Premium (comparando versões):



cliente_id,nome,segmento_original,segmento_atual
CLI00001,Cliente 1,Empresarial,Premium
CLI00002,Cliente 2,Empresarial,Premium
CLI00003,Cliente 3,Residencial,Premium


In [0]:
# Consultar dados em um momento específico no tempo
from datetime import datetime, timedelta

print("🕒 Consultando dados por timestamp\n")

# Obter timestamp da primeira versão
history_df = spark.sql("DESCRIBE HISTORY clientes")
first_timestamp = history_df.orderBy("version").first()["timestamp"]

print(f"Timestamp da primeira versão: {first_timestamp}")
print(f"\nConsultando dados como estavam naquele momento...\n")

# Consultar usando timestamp
query = f"""
SELECT 
  segmento,
  COUNT(*) as total_clientes
FROM clientes TIMESTAMP AS OF '{first_timestamp}'
GROUP BY segmento
ORDER BY total_clientes DESC
"""

display(spark.sql(query))

🕒 Consultando dados por timestamp

Timestamp da primeira versão: 2026-05-13 15:20:26

Consultando dados como estavam naquele momento...



segmento,total_clientes
Residencial,336
Premium,333
Empresarial,331


### 🔄 Restaurando Versões Anteriores

O Delta Lake permite restaurar facilmente versões anteriores dos dados:

```sql
-- Restaurar para uma versão específica
RESTORE TABLE clientes TO VERSION AS OF 0;

-- Restaurar para um timestamp específico
RESTORE TABLE clientes TO TIMESTAMP AS OF '2024-01-01 10:00:00';
```

⚠️ **Nota**: Não vamos executar o RESTORE neste workshop para manter as alterações, mas esta funcionalidade é extremamente útil para:

* Recuperação de desastres
* Correção de erros em pipelines
* Testes e validações
* Auditoria e compliance

### 🎯 Benefícios do Delta Lake Demonstrados

✅ **Versionamento Automático**: Cada operação cria uma nova versão

✅ **Time Travel**: Acesso a qualquer versão histórica dos dados

✅ **Auditoria Completa**: Histórico de todas as operações (INSERT, UPDATE, DELETE)

✅ **Recuperação Rápida**: RESTORE para voltar a versões anteriores

✅ **Consultas Flexíveis**: Query por versão ou timestamp

✅ **Performance**: Otimizações automáticas e compactação

---

💡 **Próximo Passo**: Vamos criar Dashboards e Genie!

## 6. Dashboards & Genie Spaces

Agora que temos nossas Metric Views criadas, podemos usá-las para:

* **Dashboards**: Visualizações interativas e relatórios
* **Genie Spaces**: Análise conversacional com IA

Ambos utilizam as mesmas métricas definidas, garantindo consistência!

### 📊 Criando um Dashboard

#### Passo a Passo:

1. **Acesse**: Menu lateral > **Dashboards** > **Create Dashboard**
2. **Adicione as Tabelas**: clientes, planos, receita_mensal e uso_diario

Use o Genie Code para criar seus dashboards baseados nos dados desta tabela, SEJA CRIATIVO :-)

### 🧞 Criando um Genie Space

**Genie** é o assistente de IA da Databricks que permite análise conversacional de dados.

#### Passo a Passo:

1. **Acesse**: Menu lateral > **Genie Spaces** > **Create Genie Space**

2. **Configure**:
   * **Nome**: "Análise Telefonica"
   * **Descrição**: "Análise de receita, uso e clientes da Telefonica"

3. **Adicione as Metric Views**:
   * `metricas_receita`
   * `metricas_uso`
   * `metricas_clientes`

4. **Adicione Instruções** (opcional):
   ```
   Este espaço contém dados de telecomunicações da Telefonica.
   
   - Receita: faturamento, ticket médio, inadimplência
   - Uso: consumo de dados, minutos de voz, SMS
   - Clientes: segmentação, ARPU, distribuição geográfica
   
   Valores monetários em BRL (Reais).
   ```


### 💬 Exemplos de Perguntas para o Genie

Após criar o Genie Space, você pode fazer perguntas em linguagem natural:

#### Perguntas sobre Receita:
* "Qual foi a receita total no último mês?"
* "Mostre a evolução da receita nos últimos 3 meses"
* "Qual tipo de plano gera mais receita?"
* "Qual é a taxa de inadimplência atual?"
* "Compare receita base vs receita de excedentes"

#### Perguntas sobre Uso:
* "Quantos clientes ativos temos?"
* "Qual é o consumo médio de dados por cliente?"
* "Mostre o consumo total de dados por mês"
* "Quantos minutos de voz foram usados este mês?"

#### Perguntas sobre Clientes:
* "Quantos clientes temos por segmento?"
* "Qual cidade tem o maior ARPU?"
* "Mostre a distribuição de clientes por tempo de cadastro"
* "Qual segmento consome mais dados?"

#### Perguntas Complexas:
* "Qual é a correlação entre consumo de dados e receita?"
* "Compare o ARPU entre clientes novos e antigos"
* "Identifique tendências de crescimento na receita"
* "Quais segmentos têm maior taxa de inadimplência?"

---


## 🎉 Parabéns! Workshop Concluído

### O que aprendemos:

✅ **Unity Catalog**
* Estrutura hierárquica (Catalog > Schema > Tables)
* Metadados enriquecidos e documentação
* Governança e controle de acesso
* Information Schema para consultas de metadados

✅ **Delta Lake**
* Formato ACID para data lakes
* Time Travel para consultas históricas
* Versionamento automático
* Auditoria completa de operações

✅ **Dashboards & Genie**
* Visualizações interativas
* Análise conversacional com IA
* Consistência de métricas

---

👋 **Obrigado por participar do Workshop Databricks - Telefonica!**

## 🧹 Limpeza (Opcional)

Se desejar remover os recursos criados neste workshop:

```python
# ATENÇÃO: Isso irá deletar TODOS os dados criados!
# Descomente as linhas abaixo apenas se tiver certeza

# spark.sql(f"DROP SCHEMA IF EXISTS {catalog_name}.{schema_name} CASCADE")
# spark.sql(f"DROP CATALOG IF EXISTS {catalog_name} CASCADE")
# print(f"✅ Catálogo {catalog_name} removido com sucesso!")
```

⚠️ **Importante**: 
* Esta ação é **irreversível**
* Todos os dados, e tabelas serão deletados
* Use apenas em ambientes de teste/desenvolvimento